In [31]:
import pandapipes as pp
from tespy.components import Compressor,SimpleHeatExchanger,CycleCloser, Valve, HeatExchanger,Condenser, Sink, Source
from tespy.connections import Connection
from tespy.networks import Network
import numpy as np
from tespy.tools import UserDefinedEquation


class Bidirectional_W_to_WHeatPump:

    def __init__( self,name,net,refrigerant,HC_ext_id,HC_inj_id):
        self.name = name
        self.refrigerant = refrigerant
        self.HC_ext_id=HC_ext_id
        self.HC_inj_id=HC_inj_id
        self.net=net
        
    def _build_tespy_Cycles(self):
        def my_ude(ude):
                            return ude.conns[0].calc_T_dew() +5-ude.conns[1].calc_T()
        def my_ude_dependents(ude):
            c1, c2 = ude.conns
            return [c1.p,c1.h, c2.p,c2.h]
        #Heating net 
        self.nw_Heating_net = Network()
        self.nw_Heating_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Heating_net_compressor = Compressor("compresor")
        self.Heating_net_condenser = Condenser("condensador")
        self.Heating_net_valve = Valve("valvula_expansion")
        self.Heating_net_evaporator = HeatExchanger("evaporador")
        self.Heating_net_cc=CycleCloser('CycleCloser')
        self.Heating_net_cc_2=CycleCloser('CycleCloser_Consumer_side')
        self.Heating_net_Consumer=SimpleHeatExchanger("Consumer")
        self.Heating_net_source=Source("source")
        self.Heating_net_sink=Sink("Sink")
        self.Heating_net_c0=Connection(self.Heating_net_valve, 'out1', self.Heating_net_cc, 'in1', label='0')
        self.Heating_net_c1 = Connection(self.Heating_net_cc, 'out1', self.Heating_net_evaporator, 'in2', label='1')
        self.Heating_net_c2 = Connection(self.Heating_net_evaporator, 'out2', self.Heating_net_compressor, 'in1', label='2')
        self.Heating_net_c3 = Connection(self.Heating_net_compressor, 'out1', self.Heating_net_condenser, 'in1', label='3')
        self.Heating_net_c4 = Connection(self.Heating_net_condenser, 'out1', self.Heating_net_valve, 'in1', label='4')
        self.Heating_net_c5 = Connection(self.Heating_net_evaporator, 'out1', self.Heating_net_Consumer, 'in1', label='5')
        self.Heating_net_c6 = Connection(self.Heating_net_Consumer, 'out1', self.Heating_net_cc_2, 'in1', label='6')
        self.Heating_net_c7 = Connection(self.Heating_net_cc_2, 'out1', self.Heating_net_evaporator, 'in1', label='7')  # Corrección de puertos
        # 5. Circuito de la Red (Ahora actúa como SUMIDERO de calor -> va al Condensador)
        self.Heating_net_c8 = Connection(self.Heating_net_source, 'out1', self.Heating_net_condenser, 'in2', label='8')
        self.Heating_net_c9 = Connection(self.Heating_net_condenser, 'out2', self.Heating_net_sink, 'in1', label='9')
        self.nw_Heating_net.add_conns(self.Heating_net_c0,  self.Heating_net_c1,  self.Heating_net_c2,  self.Heating_net_c3,  self.Heating_net_c4 , self.Heating_net_c5,  self.Heating_net_c6, self.Heating_net_c7, self.Heating_net_c8,self.Heating_net_c9)
        ude = UserDefinedEquation(
                'my ude', my_ude, my_ude_dependents, conns=[self.Heating_net_c1, self.Heating_net_c2])
        self.nw_Heating_net.add_ude(ude)
        #Cooling net
        self.nw_Cooling_net = Network()
        self.nw_Cooling_net.units.set_defaults(temperature="degC", pressure="bar",pressure_difference="bar",  enthalpy="J/kg", heat="W", power="W")
        self.Cooling_net_compressor = Compressor("compresor")
        self.Cooling_net_condenser = Condenser("condensador")
        self.Cooling_net_valve = Valve("valvula_expansion")
        self.Cooling_net_evaporator = HeatExchanger("evaporador")
        self.Cooling_net_cc=CycleCloser('CycleCloser')
        self.Cooling_net_cc_2=CycleCloser('CycleCloser_Consumer_side')
        self.Cooling_net_Consumer=SimpleHeatExchanger("Consumer")
        self.Cooling_net_source=Source("source")
        self.Cooling_net_sink=Sink("Sink")
        self.Cooling_net_c0=Connection(self.Cooling_net_valve, 'out1', self.Cooling_net_cc, 'in1', label='0')
        self.Cooling_net_c1 = Connection(self.Cooling_net_cc, 'out1', self.Cooling_net_evaporator, 'in2', label='1')
        self.Cooling_net_c2 = Connection(self.Cooling_net_evaporator, 'out2', self.Cooling_net_compressor, 'in1', label='2')
        self.Cooling_net_c3 = Connection(self.Cooling_net_compressor, 'out1', self.Cooling_net_condenser, 'in1', label='3')
        self.Cooling_net_c4 = Connection(self.Cooling_net_condenser, 'out1', self.Cooling_net_valve, 'in1', label='4')
        self.Cooling_net_c5=Connection(self.Cooling_net_condenser, 'out2',self.Cooling_net_Consumer, 'in1', label='5')
        self.Cooling_net_c6=Connection(self.Cooling_net_Consumer, 'out1',self.Cooling_net_cc_2 , 'in1', label='6')
        self.Cooling_net_c7=Connection(self.Cooling_net_cc_2, 'out1',self.Cooling_net_condenser , 'in2', label='7')
        self.Cooling_net_c8=Connection(self.Cooling_net_source, 'out1',self.Cooling_net_evaporator , 'in1', label='8')
        self.Cooling_net_c9=Connection(self.Cooling_net_evaporator, 'out1',self.Cooling_net_sink , 'in1', label='9')
        self.nw_Cooling_net.add_conns(self.Cooling_net_c0,  self.Cooling_net_c1,  self.Cooling_net_c2,  self.Cooling_net_c3,  self.Cooling_net_c4 , self.Cooling_net_c5,  self.Cooling_net_c6, self.Cooling_net_c7, self.Cooling_net_c8,self.Cooling_net_c9)
        ude = UserDefinedEquation(
        'my ude', my_ude, my_ude_dependents, conns=[self.Cooling_net_c1, self.Cooling_net_c2])
        self.nw_Cooling_net.add_ude(ude)
        

    def solve_cycle(self,mode,Q_know,eta_s,T_nework,T_cons):
        #Dt= Heating network 
        self.mode=mode
        T_cons_in = T_cons[0]
        T_cons_out = T_cons[1]
        T_DH_in = T_nework[0]
        T_DH_out = T_nework[1]
        if self.mode=="COOLING_NET":
            self.Cooling_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Cooling_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Cooling_net_compressor.set_attr(eta_s=eta_s)
            self.Cooling_net_c2.set_attr(fluid={self.refrigerant: 1})
            # 6. Parámetros del Consumidor 
            self.Cooling_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Cooling_net_c7.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Cooling_net_c8.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            self.Cooling_net_c9.set_attr(T=T_DH_out)
            self.Cooling_net_Consumer.set_attr(Q=Q_know)
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Cooling_net._set_p_range([p_triple, p_high])
            self.nw_Cooling_net._set_h_range([h_min,h_max])
            self.nw_Cooling_net.solve('design')
        elif self.mode=="HEATING_NET":
            self.Heating_net_evaporator.set_attr(pr1=1, pr2=1, ttd_l=5)
            self.Heating_net_condenser.set_attr(pr1=1, pr2=1, ttd_u=5)
            self.Heating_net_compressor.set_attr(eta_s=eta_s)
            self.Heating_net_c2.set_attr(fluid={self.refrigerant: 1})
            # 6. Parámetros del Consumidor 
            self.Heating_net_c5.set_attr(T=T_cons_out, p=3, fluid={"water": 1})
            self.Heating_net_c7.set_attr(T=T_cons_in)
            # 7. Parámetros de la Red de Distrito (Recibe calor, se calienta de 50 °C a 60 °C)
            self.Heating_net_c8.set_attr(T=T_DH_in, p=2.5, fluid={"water": 1})
            self.Heating_net_c9.set_attr(T=T_DH_out)
            self.Heating_net_Consumer.set_attr(Q=Q_know)
            import CoolProp.CoolProp as CP
            T_triple = CP.Props1SI("Ttriple", self.refrigerant)        # Triple point temperature (K)
            p_triple = CP.Props1SI("ptriple", self.refrigerant)        # Triple point pressure (Pa)
            T_critical = CP.Props1SI("T_critical", self.refrigerant)  # Critical temperature (K)
            p_critical = CP.Props1SI("p_critical", self.refrigerant)  # Critical pressure (Pa)
            h_min = CP.PropsSI("H", "T", T_triple + 0.1, "Q", 0, self.refrigerant)
            T_max_K =  T_critical*0.9
            p_high = min(p_critical * 0.9, 30e5) 
            h_max = CP.PropsSI("H", "T", T_max_K, "P", p_high, self.refrigerant)
            self.nw_Heating_net._set_p_range([p_triple, p_high])
            self.nw_Heating_net._set_h_range([h_min,h_max])
            self.nw_Heating_net.solve('design')
    def interaction_simulation(self,dT_water):
        if self.mode=="COOLING_NET":
            self.net.heat_consumer.at[self.HC_ext_id, "qext_w"] =abs( self.Cooling_net_evaporator.Q.val)
            self.net.heat_consumer.at[self.HC_ext_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False
            
        elif self.mode=="HEATING_NET":
            self.net.heat_consumer.at[self.HC_inj_id, "qext_w"] = self.Heating_net_condenser.Q.val
            self.net.heat_consumer.at[self.HC_inj_id,"deltat_k"]=dT_water
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False

        else: 
            self.net.heat_consumer.at[self.HC_ext_id, "in_service"] =False
            self.net.heat_consumer.at[self.HC_inj_id, "in_service"] =False

    def get_main_results(self):
        if self.mode=="COOLING_NET":
            results = {
                "Q_consumer": self.Cooling_net_Consumer.Q.val,
                "Q_network": self.Cooling_net_evaporator.Q.val,
                "COP": self.Cooling_net_compressor.P.val / abs(self.Cooling_net_Consumer.Q.val)
            }
        elif self.mode=="HEATING_NET":
            results = {
                "Q_consumer": self.Heating_net_Consumer.Q.val,
                "Q_network": self.Heating_net_evaporator.Q.val,
                "COP": self.Heating_net_compressor.P.val / self.Heating_net_Consumer.Q.val
            }
        else:
            results = {}
        return results

    def get_results_solver(self):
        if self.mode=="COOLING_NET":
            self.nw_Cooling_net.print_results()
        elif self.mode=="HEATING_NET":
            self.nw_Heating_net.print_results()
        else:
            return None


In [32]:
net = pp.create_empty_network(fluid="water")
# Nudos de la Central / Fuente
j_fuente_ida = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Fuente_Ida")
j_nodo_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_1_ida")
j_nodo_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="nodo_2_ida") 
j_HP_1 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_Ida")
j_HP_2 = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_Ida")
#Return
j_fuente_ret = pp.create_junction(net, pn_bar=1.5, tfluid_k=333.15, name="Fuente_Retorno")
j_nodo_1_ret = pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Node_1_ret")
j_nodo_2_ret=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="Nodo_2_ret") 
j_HP_ret_1= pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_1_ret")
j_HP_ret_2=pp.create_junction(net, pn_bar=5.0, tfluid_k=353.15, name="HP_2_ret")

#Plant-Nodo
u=0.35 / (np.pi * 0.15)
#Planta 1
pipe_ida_1 = pp.create_pipe_from_parameters(
    net, from_junction=j_fuente_ida, to_junction=j_nodo_1,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u, text_k=273.2, name="Tubo_Ida_Plant_Storage_ida",k_mm=0.1*1000
)

pipe_retorno_1= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1_ret, to_junction=j_fuente_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_RetornoPlant_Storage_ret",k_mm=0.1*1000)

#Nodo-nodo
pipe_ida_2 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_1, to_junction=j_nodo_2,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_Storage_ida_nodo_1",k_mm=0.1*1000
)
pipe_retorno_2= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2_ret, to_junction=j_nodo_1_ret,
    length_km=0.0589, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_Storage_ret_nodo_1",k_mm=0.1*1000
)

#Node HP
#Node_Cons1

pipe_ida_3= pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_1,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_1_cons_1_ida",k_mm=0.1*1000
)
pipe_retorno_3= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_1, to_junction=j_nodo_2_ret,
    length_km=0.0376, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_1_cons_1_ret",k_mm=0.1*1000
)
#Node_Cons2
pipe_ida_4 = pp.create_pipe_from_parameters(
    net, from_junction=j_nodo_2, to_junction=j_HP_2,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Ida_nodo_2_cons_2_ida",k_mm=0.1*1000
)
pipe_retorno_4= pp.create_pipe_from_parameters(
    net, from_junction=j_HP_ret_2, to_junction=j_nodo_2_ret,
    length_km=0.0302, inner_diameter_mm=150, u_w_per_m2k=u,  text_k=273.2, name="Tubo_Retorno_nodo_2_cons_2_ret",k_mm=0.1*1000
)

#Heat Pumps
#Heat pump: extraction
HP_1_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_1 ,
    to_junction=j_HP_ret_1,
    qext_w=150000,
    deltat_k=50,
    name="HP_1_EXT"
)
#Heat mump 1: Injection
HP_1_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_1,
    to_junction=j_HP_1,
    qext_w=-150000,
    deltat_k=50,
    name="HP_1_INJ"
)
#Heat pump: extraction
HP_2_Ext=pp.create_heat_consumer(
    net,
    from_junction=j_HP_2 ,
    to_junction=j_HP_ret_2,
    qext_w=150000,
    deltat_k=50,
    name="HP_2_EXT"
)
#Heat mump 1: Injection
HP_2_inj=pp.create_heat_consumer(
    net,
    from_junction=j_HP_ret_2,
    to_junction=j_HP_2,
    qext_w=150000,
    deltat_k=-50,
    name="HP_2_INJ"
)
#Plant: 
Plant_1=pp.create_circ_pump_const_pressure(net,flow_junction=j_fuente_ida,return_junction=j_fuente_ret,p_flow_bar=3,plift_bar=0.5,t_flow_k=340 ,name='Grid')


In [33]:
T_network_inital_guess_1=[35,25]
T_network_inital_guess_2=[50,60]


In [34]:
HP_1=Bidirectional_W_to_WHeatPump(name="heat_pump_1",refrigerant="R134a",net=net,HC_ext_id=HP_1_Ext,HC_inj_id=HP_1_inj)
HP_2=Bidirectional_W_to_WHeatPump(name="heat_pump_2",refrigerant="R134a",net=net,HC_ext_id=HP_2_Ext,HC_inj_id=HP_2_inj)
HP_1._build_tespy_Cycles()
HP_2._build_tespy_Cycles()

In [ ]:
T_network_HP_1=T_network_inital_guess_1
T_network_HP_2=T_network_inital_guess_2
T_network_HP_1_loop=T_network_inital_guess_1
T_network_HP_2_loop=T_network_inital_guess_2
tolerance=1E-6
error_1=10
error_2=10
while error_1>tolerance or error_2>tolerance:
    HP_1.solve_cycle(mode="COOLING_NET",Q_know=-15000,eta_s=0.95,T_nework=T_network_HP_1,T_cons=[50,60])
    HP_2.solve_cycle(mode="HEATING_NET",Q_know=1500,eta_s=0.95,T_nework=T_network_HP_2,T_cons=[35,25])
    HP_1.interaction_simulation(dT_water=5)
    HP_2.interaction_simulation(dT_water=-10)
    pp.pipeflow(net,mode="bidirectional")
    T_network_HP_1=[]
    T_network_HP_2=[]
    T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_from_k"]-273.15))
    T_network_HP_1.append(float(net.res_heat_consumer.at[HP_1.HC_ext_id,"t_to_k"]-273.15))
    T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_from_k"]-273.15))
    T_network_HP_2.append(float(net.res_heat_consumer.at[HP_2.HC_inj_id,"t_to_k"]-273.15))
    error_1=abs(T_network_HP_1_loop[0]-T_network_HP_1[0])
    error_2=abs(T_network_HP_2_loop[0]-T_network_HP_2[0])
    print(error_1)
    print(error_2)
    T_network_HP_1_loop=T_network_HP_1
    T_network_HP_2_loop=T_network_HP_2
    


 iter  | residual   | progress   | massflow   | pressure   | enthalpy   | fluid      | energy     | component  
-------+------------+------------+------------+------------+------------+------------+------------+------------
 1     | 2.05e+05   | 7 %        | 5.03e+01   | 1.61e+06   | 1.55e+06   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 2     | 9.41e+06   | 0 %        | 1.80e+02   | 3.16e+05   | 1.12e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 3     | 5.36e+06   | 0 %        | 1.28e+02   | 1.45e+05   | 3.50e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 4     | 5.91e+05   | 2 %        | 4.62e+00   | 3.85e+04   | 2.74e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 5     | 5.37e+05   | 2 %        | 3.30e+01   | 1.84e+05   | 2.12e+05   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 6     | 1.64e+06   | 0 %        | 3.32e+01   | 3.99e+04   | 5.79e+02   | 0.00e+00   | 0.00e+00   | 0.00e+00   
 7     | 6.57e+03   | 24 %       | 2.04e-01   | 1.15e+03   | 3.42e+01   | 0.00e+00   | 0.00e+00   | 0.0

In [36]:
HP_1.get_main_results()

{'Q_consumer': -15000.0,
 'Q_network': -14482.73479502994,
 'COP': 0.03448434699800265}

In [37]:
HP_1.get_results_solver()


##### RESULTS (SimpleHeatExchanger) #####
+----------+-----------+----------+----------+-----------+----------+
|          |         Q |       pr |       dp |   zeta_d4 |     zeta |
|----------+-----------+----------+----------+-----------+----------|
| Consumer | -1.50e+04 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
+----------+-----------+----------+----------+-----------+----------+
##### RESULTS (CycleCloser) #####
+---------------------------+------------------+-------------------+
|                           |   mass_deviation |   fluid_deviation |
|---------------------------+------------------+-------------------|
| CycleCloser               |         0.00e+00 |          0.00e+00 |
| CycleCloser_Consumer_side |         0.00e+00 |          0.00e+00 |
+---------------------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-------

In [38]:
HP_2.get_results_solver()


##### RESULTS (SimpleHeatExchanger) #####
+----------+----------+----------+----------+-----------+----------+
|          |        Q |       pr |       dp |   zeta_d4 |     zeta |
|----------+----------+----------+----------+-----------+----------|
| Consumer | 1.50e+03 | 1.00e+00 | 0.00e+00 |  0.00e+00 | 0.00e+00 |
+----------+----------+----------+----------+-----------+----------+
##### RESULTS (CycleCloser) #####
+---------------------------+------------------+-------------------+
|                           |   mass_deviation |   fluid_deviation |
|---------------------------+------------------+-------------------|
| CycleCloser               |         0.00e+00 |          0.00e+00 |
| CycleCloser_Consumer_side |         0.00e+00 |          0.00e+00 |
+---------------------------+------------------+-------------------+
##### RESULTS (Compressor) #####
+-----------+----------+----------+-----------+----------+
|           |        P |       pr |        dp |    eta_s |
|-----------+

In [39]:
net.res_circ_pump_pressure

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.5,3.0,332.341873,340.0,340.0,0.646018,-0.646018,0.000658,-7.658127,20713.14003


In [40]:
net.res_heat_consumer

,p_from_bar,p_to_bar,t_from_k,t_to_k,t_outlet_k,mdot_from_kg_per_s,mdot_to_kg_per_s,vdot_m3_per_s,deltat_k,qext_w
0,2.996673,2.503319,338.524238,333.524238,333.524238,0.691854,-0.691854,0.000705,5.000000,14482.734795
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.374238,-150000.000000
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.881005,150000.000000
3,2.502426,2.997567,330.031005,340.031005,340.031005,0.045836,-0.045836,0.000047,-10.000000,-1918.874021
